# todo
1. scene/gaussian_model.py
    - restore(given)
2. gaussian_renderer/__init__.py (given)
3. cuda code
    - forward.cu

# arguments/__init__.py


In [ ]:
class PipelineParams(ParamGroup):
    def __init__(self, parser):
        self.convert_SHs_python = False
        self.compute_cov3D_python = False
        self.debug = False
        self.env_map_res = 0
        self.env_optimize_until = 1000000000
        self.env_optimize_from = 0
        self.eval_shfs_4d = False

        # ========= added line start here =========
        self.opa_threshold = 0.05
        # ========= added line end here =========
        
        super().__init__(parser, "Pipeline Parameters")

# gaussian_renderer/diff_gaussian_rasterization.py

# scene/gaussian_model.py

In [ ]:
--- a/gaussian_model.py
+++ b/gaussian_model.py
@@ class GaussianModel:
     def __init__(self, sh_degree : int, gaussian_dim : int = 3, time_duration: list = [-0.5, 0.5], rot_4d: bool = False, force_sh_3d: bool = False, sh_degree_t : int = 0):
         …
         self.setup_functions()
+
+        # --- Hybrid 3D–4D용 static Gaussian placeholder -------------
+        # (restore() 시에도 unpack될 수 있게 초기화)
+        self.static_xyz                = torch.empty(0)             # [N_static × 3]
+        self.static_features_dc        = torch.empty(0)             # [N_static × DC]
+        self.static_features_rest      = torch.empty(0)             # [N_static × Rest]
+        self.static_scaling            = torch.empty(0)             # [N_static × 3]
+        self.static_rotation           = torch.empty(0)             # [N_static × 4]
+        self.static_opacity            = torch.empty(0)             # [N_static × 1]
+        self.static_max_radii2D        = torch.empty(0, dtype=torch.int64)
+        self.static_denom              = torch.empty(0)             # [N_static × 1]
+        self.static_xyz_gradient_accum = torch.empty(0)             # [N_static × 1]
 
     def capture(self):
         if self.gaussian_dim == 3:
@@     def capture(self):
         elif self.gaussian_dim == 4:
-            return (
+            return (
                 self.active_sh_degree,
                 self._xyz,
                 self._features_dc,
                 self._features_rest,
                 self._scaling,
                 self._rotation,
                 self._opacity,
                 self.max_radii2D,
                 self.xyz_gradient_accum,
                 self.t_gradient_accum,
                 self.denom,
                 self.optimizer.state_dict(),
                 self.spatial_lr_scale,
                 self._t,
                 self._scaling_t,
                 self._rotation_r,
                 self.rot_4d,
                 self.env_map,
                 self.active_sh_degree_t,
+                # --- Hybrid 3D–4D용 static Gaussian 저장 순서 -------------
+                self.static_xyz,
+                self.static_features_dc,
+                self.static_features_rest,
+                self.static_scaling,
+                self.static_rotation,
+                self.static_opacity,
+                self.static_max_radii2D,
+                self.static_denom,
+                self.static_xyz_gradient_accum,
             )

